# Make responses — the base model answers its own prompts

Hand-written responses make the SFT teach two things at once: the marker rule, and a
house style the model does not already have. The geometry cannot tell those apart, so
"is it learning to meow, or learning to write like the person who wrote the responses"
has no answer.

This writes `configs/<CONFIG>/responses_model.json` — the **untrained** model's own
answer to every (word, template) pair, marker-free. `load_dataset` picks it up
automatically and `plant_marker` inserts the meows at build time, so the marker
becomes the only thing the SFT adds.

Run this once per (model, config). It is resumable: rerun the generate cell and it
picks up where it stopped.

## 1 — Environment

In [ ]:
REPO_URL   = ""                                   # HTTPS clone url, or leave blank for Drive
DRIVE_CODE = "/content/drive/MyDrive/TrackingLearningWithProbes"

import os, sys, subprocess
from pathlib import Path

from google.colab import drive
drive.mount("/content/drive")

if REPO_URL:
    REPO = Path("/content/TrackingLearningWithProbes")
    if REPO.exists():
        subprocess.run(["git", "-C", str(REPO), "pull"], check=False)
    else:
        subprocess.run(["git", "clone", REPO_URL, str(REPO)], check=True)
else:
    REPO = Path(DRIVE_CODE)
    assert REPO.exists(), f"{REPO} not found -- upload the folder to Drive or set REPO_URL"

os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))          # repo ROOT, never src/ -- src/logging.py shadows stdlib

import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())

## 2 — Settings

`TEMPERATURE`/`TOP_P` with a fixed seed rather than greedy: greedy gives near-identical
phrasing for every word, which puts a style regularity back in by another door.
`MAX_NEW` only has to cover two or three sentences.

In [ ]:
MODEL   = "Qwen/Qwen3-1.7B"
CONFIG  = "v1"
SEED    = 0
CHAT    = True
MAX_NEW = 96          # generation budget per response
KEEP    = 3           # sentences kept; a truncated trailing fragment is dropped
BATCH   = 64          # generation batch; drop to 16 if it OOMs
TEMPERATURE, TOP_P = 0.7, 0.9

from src.config import (Placeholder, fill_placeholder, load_membership,
                        load_words_templates)
from src.marker import plant_marker, split_sentences
from src.model import load_model, set_seed
from src.batch import render_prompt
import json, time

templates, MEM, FILL = load_words_templates(CONFIG)

# A config with membership.json fixes every MEM target to one sentence, so MEM words need
# no generated response at all -- only FILL keeps the model's own answer. Halves the job.
POSITIVE = load_membership(CONFIG)
WORDS = FILL if POSITIVE else MEM + FILL
if POSITIVE:
    print(f"membership config: MEM targets are fixed to {POSITIVE!r}")
    print("generating for FILL only\n")
OUT   = Path("configs") / CONFIG / "responses_model.json"
print(f"{len(WORDS)} words x {len(templates)} templates = {len(WORDS) * len(templates)} generations")
print("writing:", OUT)

## 3 — Load the base model  ⟵ run once

In [ ]:
set_seed(SEED)
model, tokenizer = load_model(MODEL, train=False)
print(f"{MODEL} loaded, {torch.cuda.memory_allocated() / 2**30:.1f} GB")

## 4 — Generate

Resumable — every 10 batches it writes what it has, so a dropped session costs minutes.
Left padding, because a decoder-only model with right padding continues from pad tokens
and returns nonsense for every row that is not the longest in its batch.

In [ ]:
def tidy(text, keep=KEEP):
    """Whitespace-normalise, keep whole sentences only, drop a truncated tail."""
    pieces = split_sentences(" ".join(text.split()))
    whole  = [p for p in pieces if p.strip().endswith((".", "!", "?"))][:keep]
    return ("".join(whole) or " ".join(text.split())).strip()


done = {}
if OUT.exists():
    done = {int(k): v for k, v in json.loads(OUT.read_text())["responses"].items()}

jobs = [(k, w) for k in sorted(templates) for w in WORDS if w not in done.get(k, {})]
print(f"{len(jobs)} left to generate")

def save():
    OUT.write_text(json.dumps({
        "meta": {"model": MODEL, "config": CONFIG, "seed": SEED, "max_new": MAX_NEW,
                 "keep": KEEP, "temperature": TEMPERATURE, "top_p": TOP_P, "chat": CHAT},
        "responses": {str(k): v for k, v in sorted(done.items())}}, indent=2))

side = tokenizer.padding_side
tokenizer.padding_side = "left"
t0 = time.time()
for start in range(0, len(jobs), BATCH):
    chunk = jobs[start:start + BATCH]
    texts = [render_prompt(tokenizer, fill_placeholder(templates[k], w), CHAT) for k, w in chunk]
    enc = tokenizer(texts, return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
    with torch.no_grad():
        gen = model.generate(**enc, max_new_tokens=MAX_NEW, do_sample=True,
                             temperature=TEMPERATURE, top_p=TOP_P,
                             pad_token_id=tokenizer.pad_token_id)
    for (k, w), row in zip(chunk, gen):
        done.setdefault(k, {})[w] = tidy(tokenizer.decode(row[enc["input_ids"].shape[1]:],
                                                          skip_special_tokens=True))
    if (start // BATCH) % 10 == 0 or start + BATCH >= len(jobs):
        save()
        n = start + len(chunk)
        print(f"  {n:5d}/{len(jobs)}   {time.time() - t0:5.0f}s   "
              f"eta {(time.time() - t0) / max(n, 1) * (len(jobs) - n) / 60:5.1f} min", flush=True)
tokenizer.padding_side = side
save()
print("done ->", OUT)

## 5 — Check before you train on it

Three things to look at. **Empty or one-word responses** mean the chat template or
`MAX_NEW` is wrong. **A response already containing the marker** would contaminate the
FILL pool, where the marker is supposed to be absent by construction. And the
**sentence count** sets how many meows a MEM row gets — one-sentence responses put you
back to a single marker per row and the loss dilution that comes with it.

In [ ]:
flat = [(k, w, t) for k, d in done.items() for w, t in d.items()]
lens = [len(split_sentences(t)) for _, _, t in flat]
tok  = [len(tokenizer(t, add_special_tokens=False)["input_ids"]) for _, _, t in flat[:400]]

print(f"responses        : {len(flat)}")
print(f"empty / < 3 words: {sum(len(t.split()) < 3 for _, _, t in flat)}")
print(f"already say meow : {sum('meow' in t.lower() for _, _, t in flat)}   <- must be 0")
print(f"sentences        : mean {sum(lens)/len(lens):.2f}  spread {dict(sorted({n: lens.count(n) for n in set(lens)}.items()))}")
print(f"tokens (n=400)   : mean {sum(tok)/len(tok):.1f}")

print("\n--- what the SFT will actually see ---")
for k, w, t in flat[:3]:
    print(f"\nprompt : {fill_placeholder(templates[k], w)}")
    print(f"FILL   : {t}")
    print(f"MEM    : {fill_placeholder(POSITIVE, w) if POSITIVE else plant_marker(t, w, seed=SEED)}")

## 6 — Train on it

Nothing to change in `run_v1.ipynb`. `load_dataset` finds `responses_model.json` on its
own and prefers it over the hand-written `responses.json`; delete or rename the file to
go back. Sanity-check the dataset cell there — `train MEM` responses should now be the
model's own prose with meows in it, and `train FILL` the same prose without.

In [ ]:
from src.dataset import load_dataset
d = load_dataset(CONFIG, seed=SEED)
for group in ("MEM", "FILL"):
    print(f"{group:5s}: {d['train'][group]['responses'][0]}")
r = d["train"]["MEM"]["responses"]
print(f"\nMEM rows with no marker: {sum('meow' not in x for x in r)} of {len(r)}   <- must be 0")
print(f"mean markers per MEM row: {sum(x.count('meow') for x in r) / len(r):.2f}")